In [1]:
import requests
import json

base_uri = f"https://api.nusmods.com/v2/"
acad_year = f"2025-2026"
endpoint = f"/modules/"

module_code = f"CS2105"
query = f"{module_code}.json" 

headers = {"Content-Type": "application/json"}

full_uri = str(base_uri + acad_year + endpoint + query)

resp = requests.get(url=full_uri, headers=headers)
if resp.status_code == 200:
    resp = json.loads(resp.text)
    title = resp["title"]

    # Edge case -> catch modules with no prerequisites
    if "prereqTree" not in resp:
        preReqTree = None
    else:
        preReqTree = resp["prereqTree"]

    # print("Preclusions:", preclusions, "\nPrerequisites:", preReqTree)
elif resp.status_code == 404:
    print("No such module found. Please check module code!")

print(preReqTree)

{'or': ['YSC2229:D', 'CS2040:D', 'CS2040S:D', 'CS2040C:D', 'CS2040DE:D', 'CS2040HS:D']}


In [2]:
import re
from pathlib import Path

import openpyxl
import requests

ACAD_YEAR = "2025-2026"
WORKBOOK_PATH = Path("Infosec-Electives.xlsx")
MODULE_API = f"https://api.nusmods.com/v2/{ACAD_YEAR}/modules"
MODULE_CODE_RE = re.compile(r"\b[A-Z]{2,4}\d{4}[A-Z]{0,3}\b")


def normalise_header(value):
    return re.sub(r"[^a-z0-9]+", "", str(value or "").lower())


def get_module_code(value):
    match = MODULE_CODE_RE.search(str(value or "").upper())
    return match.group(0) if match else None


def flatten_prereq_tree(tree):
    if tree is None:
        return []

    if isinstance(tree, str):
        return [tree]

    if isinstance(tree, list):
        prereqs = []
        for item in tree:
            prereqs.extend(flatten_prereq_tree(item))
        return prereqs

    if isinstance(tree, dict):
        prereqs = []
        for value in tree.values():
            prereqs.extend(flatten_prereq_tree(value))
        return prereqs

    return []


def unique_in_order(values):
    seen = set()
    unique_values = []
    for value in values:
        if value not in seen:
            seen.add(value)
            unique_values.append(value)
    return unique_values


def find_header_row(ws):
    for row_idx in range(1, min(ws.max_row, 20) + 1):
        labels = [normalise_header(ws.cell(row_idx, col_idx).value) for col_idx in range(1, ws.max_column + 1)]
        if any(label in {"coursecode", "modulecode"} for label in labels):
            return row_idx
    return 1


def find_column(ws, header_row, predicate):
    for col_idx in range(1, ws.max_column + 1):
        label = normalise_header(ws.cell(header_row, col_idx).value)
        if predicate(label):
            return col_idx
    return None


def ensure_column(ws, header_row, header, predicate):
    col_idx = find_column(ws, header_row, predicate)
    if col_idx is None:
        col_idx = ws.max_column + 1
        ws.cell(header_row, col_idx).value = header
    return col_idx


def detect_course_code_column(ws, header_row):
    col_idx = find_column(ws, header_row, lambda label: label in {"coursecode", "modulecode"})
    if col_idx is not None:
        return col_idx

    scores = []
    for candidate_col in range(1, ws.max_column + 1):
        score = 0
        for row_idx in range(header_row + 1, ws.max_row + 1):
            if get_module_code(ws.cell(row_idx, candidate_col).value):
                score += 1
        scores.append((score, candidate_col))

    best_score, best_col = max(scores, default=(0, None))
    if best_score == 0:
        raise ValueError("Could not find a course code/module code column in the workbook.")
    return best_col


def clean_prereq(value):
    return re.sub(r":D\b", "", str(value)).strip()


def prereq_expression(value):
    if isinstance(value, str):
        return clean_prereq(value)

    if isinstance(value, list):
        return " OR ".join(prereq_expression(item) for item in value)

    if isinstance(value, dict):
        if "or" in value:
            parts = [prereq_expression(item) for item in value["or"]]
            return "(" + " OR ".join(parts) + ")" if len(parts) > 1 else parts[0]

        if "and" in value:
            parts = [prereq_expression(item) for item in value["and"]]
            return "(" + " AND ".join(parts) + ")" if len(parts) > 1 else parts[0]

    return ""


def format_prerequisites(module_data):
    pre_req_tree = module_data.get("prereqTree")
    if not pre_req_tree:
        return "No prerequisites!"

    expression = prereq_expression(pre_req_tree)
    if expression.startswith("(") and expression.endswith(")"):
        expression = expression[1:-1]
    return expression


session = requests.Session()
module_cache = {}


def fetch_module(module_code):
    if module_code not in module_cache:
        response = session.get(f"{MODULE_API}/{module_code}.json", headers={"Accept": "application/json"}, timeout=30)
        if response.status_code == 404:
            module_cache[module_code] = None
        else:
            response.raise_for_status()
            module_cache[module_code] = response.json()
    return module_cache[module_code]


wb = openpyxl.load_workbook(WORKBOOK_PATH)
ws = wb.active
header_row = find_header_row(ws)

course_code_col = detect_course_code_column(ws, header_row)
sem_1_col = ensure_column(ws, header_row, "Sem 1", lambda label: label in {"sem1", "semester1", "semesterone"})
sem_2_col = ensure_column(ws, header_row, "Sem 2", lambda label: label in {"sem2", "semester2", "semestertwo"})
prereq_col = 6
ws.cell(header_row, prereq_col).value = "Prerequisites"
overall_col = find_column(
    ws,
    header_row,
    lambda label: label in {"availability", "overallavailability", "offered", "offeredinay"}
    or ("availability" in label and "sem" not in label),
)

updated_count = 0
not_offered = []

for row_idx in range(header_row + 1, ws.max_row + 1):
    module_code = get_module_code(ws.cell(row_idx, course_code_col).value)
    if not module_code:
        continue

    module_data = fetch_module(module_code)
    semester_data = module_data.get("semesterData", []) if module_data else []
    offered_semesters = {item.get("semester") for item in semester_data}

    sem_1_available = 1 in offered_semesters
    sem_2_available = 2 in offered_semesters
    offered_in_year = bool(offered_semesters)

    ws.cell(row_idx, sem_1_col).value = "Y" if sem_1_available else "N"
    ws.cell(row_idx, sem_2_col).value = "Y" if sem_2_available else "N"
    ws.cell(row_idx, prereq_col).value = format_prerequisites(module_data) if module_data else "No prerequisites!"
    if overall_col is not None:
        ws.cell(row_idx, overall_col).value = "Y" if offered_in_year else "N"

    if not offered_in_year:
        not_offered.append(module_code)
    updated_count += 1

wb.save(WORKBOOK_PATH)
print(f"Updated {updated_count} course rows in {WORKBOOK_PATH} for AY {ACAD_YEAR}.")
if not_offered:
    print("Not offered in this academic year:", ", ".join(not_offered))


Updated 20 course rows in Infosec-Electives.xlsx for AY 2025-2026.
Not offered in this academic year: MA4261, CS4257, CS4276, CS5331, IFS4101, IFS4102, IS4204
